In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

from sklearn.feature_selection import mutual_info_regression, f_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from torch.utils.data import DataLoader, TensorDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [7]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv(
    "../household_power_consumption.txt",
    sep=';',
    low_memory=False
)

# Combine Date + Time
df['datetime'] = pd.to_datetime(
    df['Date'] + ' ' + df['Time'],
    format='%d/%m/%Y %H:%M:%S'
)

df.set_index('datetime', inplace=True)
df.drop(columns=['Date', 'Time'], inplace=True)

# Replace missing markers
df.replace('?', np.nan, inplace=True)

# Remove hidden spaces
df = df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

# Convert to numeric
df = df.apply(pd.to_numeric, errors='coerce')

# Interpolate missing values
df = df.interpolate(method='time')
df = df.bfill()

# Ensure datetime index & sort
df.index = pd.to_datetime(df.index)
df = df.sort_index()

# Resample to hourly
df_hourly = df.resample('h').mean()

In [8]:
print("Hourly Dataset Shape:", df_hourly.shape)
print("\nFirst 5 rows:")
print(df_hourly.head())

print("\nMissing values after resampling:")
print(df_hourly.isna().sum())

Hourly Dataset Shape: (34589, 7)

First 5 rows:
                     Global_active_power  Global_reactive_power     Voltage  \
datetime                                                                      
2006-12-16 17:00:00             4.222889               0.229000  234.643889   
2006-12-16 18:00:00             3.632200               0.080033  234.580167   
2006-12-16 19:00:00             3.400233               0.085233  233.232500   
2006-12-16 20:00:00             3.268567               0.075100  234.071500   
2006-12-16 21:00:00             3.056467               0.076667  237.158667   

                     Global_intensity  Sub_metering_1  Sub_metering_2  \
datetime                                                                
2006-12-16 17:00:00         18.100000             0.0        0.527778   
2006-12-16 18:00:00         15.600000             0.0        6.716667   
2006-12-16 19:00:00         14.503333             0.0        1.433333   
2006-12-16 20:00:00         13.91

In [9]:
features = [
    'Global_reactive_power',
    'Voltage',
    'Global_intensity',
    'Sub_metering_1',
    'Sub_metering_2',
    'Sub_metering_3'
]

target = 'Global_active_power'

X_full = df_hourly[features]
y_full = df_hourly[target]

In [10]:
corr_values = df_hourly.corr()[target][features]
abs_corr = corr_values.abs()

corr_rank = abs_corr.sort_values(ascending=False)

print("Pearson Correlation Ranking:")
print(corr_rank)

Pearson Correlation Ranking:
Global_intensity         0.999419
Sub_metering_3           0.696104
Sub_metering_1           0.495645
Sub_metering_2           0.439244
Voltage                  0.374436
Global_reactive_power    0.307192
Name: Global_active_power, dtype: float64


In [11]:
from sklearn.feature_selection import f_regression

f_scores, p_values = f_regression(X_full, y_full)
f_series = pd.Series(f_scores, index=features)

f_rank = f_series.sort_values(ascending=False)

print("F-Test Ranking:")
print(f_rank)

F-Test Ranking:
Global_intensity         2.972748e+07
Sub_metering_3           3.251503e+04
Sub_metering_1           1.126390e+04
Sub_metering_2           8.268319e+03
Voltage                  5.639905e+03
Global_reactive_power    3.603971e+03
dtype: float64


In [12]:
from sklearn.feature_selection import mutual_info_regression

mi_scores = mutual_info_regression(X_full, y_full)
mi_series = pd.Series(mi_scores, index=features)

mi_rank = mi_series.sort_values(ascending=False)

print("Mutual Information Ranking:")
print(mi_rank)

Mutual Information Ranking:
Global_intensity         3.213132
Sub_metering_3           0.612892
Sub_metering_2           0.148375
Sub_metering_1           0.145477
Voltage                  0.139716
Global_reactive_power    0.109920
dtype: float64


In [13]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_full, y_full)

rf_importance = pd.Series(rf.feature_importances_, index=features)

rf_rank = rf_importance.sort_values(ascending=False)

print("Random Forest Importance Ranking:")
print(rf_rank)

Random Forest Importance Ranking:
Global_intensity         0.999139
Voltage                  0.000339
Global_reactive_power    0.000277
Sub_metering_3           0.000127
Sub_metering_2           0.000060
Sub_metering_1           0.000059
dtype: float64


In [14]:
combined = pd.DataFrame({
    'Correlation': abs_corr,
    'F_Test': f_series,
    'Mutual_Info': mi_series,
    'RF_Importance': rf_importance
})

combined['Average_Rank'] = combined.rank(ascending=False).mean(axis=1)

combined_sorted = combined.sort_values(by='Average_Rank')

print("\nCombined Feature Ranking:")
print(combined_sorted)


Combined Feature Ranking:
                       Correlation        F_Test  Mutual_Info  RF_Importance  \
Global_intensity          0.999419  2.972748e+07     3.213132       0.999139   
Sub_metering_3            0.696104  3.251503e+04     0.612892       0.000127   
Sub_metering_1            0.495645  1.126390e+04     0.145477       0.000059   
Sub_metering_2            0.439244  8.268319e+03     0.148375       0.000060   
Voltage                   0.374436  5.639905e+03     0.139716       0.000339   
Global_reactive_power     0.307192  3.603971e+03     0.109920       0.000277   

                       Average_Rank  
Global_intensity               1.00  
Sub_metering_3                 2.50  
Sub_metering_1                 4.00  
Sub_metering_2                 4.00  
Voltage                        4.25  
Global_reactive_power          5.25  
